# Guided Lab: End-to-End RAG Pipeline

This lab builds a real retrieval-augmented generation workflow around one public PDF.

What you will do:
- download a real PDF
- load and split it
- create embeddings with a local Hugging Face model
- store chunks in ChromaDB
- answer questions with Groq
- trace the run with LangSmith


## Why this PDF?

A technical survey-style paper gives you enough material for meaningful retrieval.

We will use the arXiv paper:
**A Comprehensive Overview of Large Language Models** (`2307.06435`).

The notebook follows the same core pattern described in LangChain’s RAG and knowledge-base docs: load documents, split them, embed them, store them, retrieve relevant chunks, and generate an answer from the retrieved context.


## 1) Install packages


In [ ]:
%pip install -qU python-dotenv requests pypdf langchain langchain-community langchain-text-splitters langchain-chroma chromadb langchain-huggingface langchain-groq sentence-transformers


## 2) Configure environment variables

Create a `.env` file with:

```env
GROQ_API_KEY=your_groq_key
LANGSMITH_API_KEY=your_langsmith_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=rag-pdf-demo
```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "true").lower() == "true"
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "rag-pdf-demo")

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("LANGSMITH_API_KEY set:", bool(LANGSMITH_API_KEY))
print("LANGSMITH_TRACING:", LANGSMITH_TRACING)
print("LANGSMITH_PROJECT:", LANGSMITH_PROJECT)


## 3) Download the PDF


In [ ]:
from pathlib import Path
import requests

PDF_URL = "https://arxiv.org/pdf/2307.06435.pdf"
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "llm_overview_2307_06435.pdf"

def download_file(url: str, dest: Path) -> Path:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Using cached file: {dest}")
        return dest

    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    with open(dest, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    print(f"Downloaded: {dest}")
    return dest

download_file(PDF_URL, PDF_PATH)


## 4) Load the PDF


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print("Pages loaded:", len(pages))
print("First page metadata:", pages[0].metadata)
print("First page preview:")
print(pages[0].page_content[:500])


## 5) Split into chunks


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
)

chunks = splitter.split_documents(pages)

print("Chunk count:", len(chunks))
print("Example chunk metadata:", chunks[0].metadata)
print("Example chunk preview:")
print(chunks[0].page_content[:700])


## 6) Create local embeddings


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready")


## 7) Store chunks in ChromaDB


In [ ]:
from langchain_chroma import Chroma
import shutil

CHROMA_DIR = Path("./chroma_data")
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llm_overview_pdf",
    persist_directory=str(CHROMA_DIR),
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Chroma directory:", CHROMA_DIR.resolve())


## 8) Inspect retrieval


In [ ]:
query = "What topics are covered in the survey paper?"
docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"--- Retrieved {i} ---")
    print("page:", doc.metadata.get("page"))
    print(doc.page_content[:500])
    print()


## 9) Create the Groq model


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

print(llm)


## 10) Build the LCEL RAG chain


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def format_docs(docs):
    return "".join(
        f"[page={doc.metadata.get('page')}] {doc.page_content}"
        for doc in docs
    )

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant that answers only from the provided PDF context. "
        "If the answer is not in the context, say you do not know."
    ),
    (
        "user",
        """Question: {question}

Context:
{context}

Answer in 4 sentences or fewer."""
    ),
])

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("LCEL RAG chain ready")


## 11) Ask questions


In [ ]:
questions = [
    "What is the main focus of this survey paper?",
    "What broad topics does the paper say LLM research covers?",
    "Why do LLMs benefit from improved context length?",
]

for q in questions:
    print("QUESTION:", q)
    print("ANSWER:", rag_chain.invoke(q))


## 12) Add LangSmith tracing


In [ ]:
from langsmith import traceable

@traceable(name="rag_pdf_question_answer")
def answer_question(question: str) -> str:
    return rag_chain.invoke(question)

print(answer_question("What does the survey paper cover?"))


## 13) Helper that prints retrieval plus answer


In [ ]:
def ask(question: str):
    retrieved = retriever.invoke(question)

    print("Retrieved chunks:")
    for i, doc in enumerate(retrieved, 1):
        print(f"[{i}] page={doc.metadata.get('page')}")
        print(doc.page_content[:300].replace("", " "))
        print()

    answer = answer_question(question)
    print("Final answer:")
    print(answer)

ask("What broad categories of work does the paper review?")


## 14) Clean up


In [ ]:
def cleanup():
    if CHROMA_DIR.exists():
        shutil.rmtree(CHROMA_DIR)
        print("Removed", CHROMA_DIR)
    else:
        print("Nothing to remove")

# cleanup()


## Key takeaways

- A real PDF makes the RAG lab meaningful.
- PDF → chunks → embeddings → Chroma → LCEL RAG is the core workflow.
- Groq provides the generation layer.
- LangSmith tracing makes the pipeline observable.


## References

- LangChain RAG: https://docs.langchain.com/oss/python/langchain/rag
- LangChain knowledge base: https://docs.langchain.com/oss/python/langchain/knowledge-base
- Groq integration: https://docs.langchain.com/oss/python/integrations/providers/groq
- Chroma integration: https://docs.langchain.com/oss/python/integrations/providers/chroma
- Trace LangChain applications: https://docs.langchain.com/langsmith/trace-with-langchain
- PDF source: https://arxiv.org/pdf/2307.06435.pdf
